<a href="https://colab.research.google.com/github/lucasmxsantos/Jogo-da-Velha.py/blob/main/CS02_machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise de Risco de Equipamentos — CS02 Machine Learning

## Contexto CRISP-DM

Este notebook percorre as seguintes fases do processo **CRISP-DM** (*Cross-Industry Standard Process for Data Mining*):

| Fase | Status | Descrição |
|------|--------|-----------|
| 1. Entendimento do Negócio | ✅ | Classificar equipamentos com risco operacional |
| 2. Entendimento dos Dados | ✅ | EDA — distribuições, correlações, padrões temporais |
| 3. Preparação dos Dados | ✅ | Limpeza, outliers (IQR), encoding de categóricas |
| 4. Modelagem | 🔍 Preview | Árvore de Decisão (aprofundado no Entregável 2) |
| 5. Avaliação | — | Próximo entregável |
| 6. Implantação | — | Próximo entregável |

**Objetivo:** classificar medições de sensores de temperatura e umidade como **risco** (`1`) ou **sem risco** (`0`), identificando proativamente equipamentos eletrônicos em ambientes comerciais que demandam manutenção preventiva.

In [ ]:
# importando as bibliotecas necessarias
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ============================================================
# DATASET SIMULADO
# Simula medicoes de sensores em 5 equipamentos eletronicos
# EQ-001: temperatura critica | EQ-003: umidade critica
# ============================================================
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

equipamentos = ['EQ-001', 'EQ-002', 'EQ-003', 'EQ-004', 'EQ-005']
sensores     = ['S1', 'S2', 'S3', 'S4', 'S5']

config = {
    'EQ-001': {'temp_mean': 38, 'temp_std': 4, 'umid_mean': 58, 'umid_std': 6},
    'EQ-002': {'temp_mean': 26, 'temp_std': 3, 'umid_mean': 55, 'umid_std': 5},
    'EQ-003': {'temp_mean': 28, 'temp_std': 3, 'umid_mean': 73, 'umid_std': 7},
    'EQ-004': {'temp_mean': 24, 'temp_std': 2, 'umid_mean': 50, 'umid_std': 5},
    'EQ-005': {'temp_mean': 31, 'temp_std': 3, 'umid_mean': 60, 'umid_std': 6},
}

data_inicio = datetime(2024, 1, 1)
registros = []

for i in range(600):
    data   = data_inicio + timedelta(hours=int(np.random.uniform(0, 8759)))
    equip  = np.random.choice(equipamentos)
    sensor = np.random.choice(sensores)
    c      = config[equip]
    temp   = round(np.random.normal(c['temp_mean'], c['temp_std']), 1)
    umid   = round(float(np.clip(np.random.normal(c['umid_mean'], c['umid_std']), 20, 100)), 1)
    registros.append({
        'data_medicao': data.strftime('%Y-%m-%d %H:%M:%S'),
        'equipamento':  equip,
        'sensor':       sensor,
        'temperatura':  temp,
        'umidade':      umid
    })

df = pd.DataFrame(registros).sort_values('data_medicao').reset_index(drop=True)

print('Dataset simulado criado com sucesso!')
print('Formato (linhas x colunas):', df.shape)
df.head(10)

## Justificativa do Dataset Simulado

O dataset original dependia de um arquivo CSV enviado manualmente (`files.upload()`), tornando o notebook **não reproduzível**. Para garantir reprodutibilidade e autonomia, os dados foram **simulados com base nos seguintes critérios**:

- **Problema real:** monitoramento de sensores (temperatura e umidade) em terminais de autoatendimento e equipamentos eletrônicos de ambientes comerciais (ex: lotéricas, bancos, varejo).
- **Referência técnica:** normas para equipamentos eletrônicos em ambientes climatizados — temperatura ideal: 18–30 °C; umidade relativa: 40–70%.
- **Perfis simulados:** cada equipamento tem um perfil de comportamento distinto, refletindo cenários reais de risco:
  - `EQ-001`: temperatura cronicamente elevada (média 38 °C) → risco térmico
  - `EQ-003`: umidade cronicamente alta (média 73%) → risco de condensação
  - `EQ-002`, `EQ-004`, `EQ-005`: perfis normais, servem como controle

> **600 registros ao longo de 12 meses**, com `np.random.seed(42)` para garantir reprodutibilidade completa.

In [ ]:
# verificando os tipos de cada coluna
print(df.dtypes)

In [ ]:
# estatisticas descritivas das colunas numericas
df.describe()

Limpeza dos dados

In [ ]:
# checando se tem valores nulos
print('Valores nulos por coluna:')
print(df.isnull().sum())

In [ ]:
# checando se tem linhas duplicadas
print('Linhas duplicadas:', df.duplicated().sum())

In [ ]:
# a coluna data_medicao esta como texto (object)
# preciso converter para o tipo datetime
df['data_medicao'] = pd.to_datetime(df['data_medicao'])
print('Tipo apos conversao:', df['data_medicao'].dtype)

In [ ]:
# extraindo mes e hora para usar na analise
df['mes'] = df['data_medicao'].dt.month
df['hora'] = df['data_medicao'].dt.hour

print('Colunas novas criadas: mes e hora')
print('Colunas atuais:', df.columns.tolist())
df.head()

In [ ]:
# ============================================================
# TRATAMENTO DE OUTLIERS — Metodo IQR
# ============================================================

def detectar_outliers_iqr(serie, nome):
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    outliers = serie[(serie < lim_inf) | (serie > lim_sup)]
    print(f'{nome}:  Q1={Q1:.1f}  Q3={Q3:.1f}  IQR={IQR:.1f}  |  Limites: [{lim_inf:.1f}, {lim_sup:.1f}]')
    print(f'  Outliers detectados: {len(outliers)} ({len(outliers)/len(serie)*100:.1f}%)\n')
    return lim_inf, lim_sup

print('=== DETECCAO DE OUTLIERS (IQR) ===\n')
lim_t_inf, lim_t_sup = detectar_outliers_iqr(df['temperatura'], 'Temperatura (C)')
lim_u_inf, lim_u_sup = detectar_outliers_iqr(df['umidade'],    'Umidade     (%)')

# Decisao: MANTER outliers — temperaturas/umidades extremas sao exatamente
# os eventos de risco que o modelo deve aprender a detectar.
# Remove-los eliminaria o sinal mais importante do dataset.

df['outlier_temp'] = ((df['temperatura'] < lim_t_inf) | (df['temperatura'] > lim_t_sup)).astype(int)
df['outlier_umid'] = ((df['umidade']    < lim_u_inf) | (df['umidade']    > lim_u_sup)).astype(int)

print(f'Medicoes com outlier de temperatura: {df["outlier_temp"].sum()}')
print(f'Medicoes com outlier de umidade:     {df["outlier_umid"].sum()}')
print(f'\nOutliers mantidos como feature adicional (outlier_temp, outlier_umid).')

In [ ]:
# valores unicos nas colunas categoricas
print('Equipamentos unicos:', df['equipamento'].unique())
print('Sensores unicos:', df['sensor'].unique())
print()
print('Quantidade de medicoes por equipamento:')
print(df['equipamento'].value_counts())

In [ ]:
# transformando equipamento em numero com mapeamento manual
mapa_equip = {'EQ-001': 1, 'EQ-002': 2, 'EQ-003': 3, 'EQ-004': 4, 'EQ-005': 5}
df['equipamento_num'] = df['equipamento'].map(mapa_equip)

# transformando sensor em numero
mapa_sensor = {'S1': 1, 'S2': 2, 'S3': 3, 'S4': 4, 'S5': 5}
df['sensor_num'] = df['sensor'].map(mapa_sensor)

print('Variaveis categoricas transformadas com sucesso!')
df[['equipamento', 'equipamento_num', 'sensor', 'sensor_num']].head(8)

## Nota sobre o Encoding Ordinal

O mapeamento `{'EQ-001': 1, ..., 'EQ-005': 5}` é um **encoding ordinal**, que implica ordem numérica entre os equipamentos (como se `EQ-005 > EQ-001`). Isso **não é verdade** — os equipamentos não têm hierarquia natural.

**Alternativas mais corretas:**
- **OneHotEncoding:** cria colunas binárias para cada categoria, sem implicar ordem. Preferível para modelos lineares e redes neurais.
- **LabelEncoder (sklearn):** atribui inteiros de forma controlada, mas ainda cria pseudo-ordem.
- **Target Encoding:** substitui pela média da variável-alvo — útil em alta cardinalidade.

**Por que mantemos ordinal aqui?**
> Árvores de Decisão e modelos baseados em divisões (Random Forest, XGBoost) **não são afetados pela pseudo-ordinalidade** — eles aprendem thresholds corretos independentemente. Para este dataset e modelo, o mapeamento manual é suficiente e interpretável. Em modelos lineares, use OneHotEncoding.

ANALISE EXPLORATORIA

In [ ]:
# histograma da temperatura
plt.figure(figsize=(8, 4))
plt.hist(df['temperatura'], bins=15, color='tomato', edgecolor='black')
plt.title('Distribuicao da Temperatura')
plt.xlabel('Temperatura (graus C)')
plt.ylabel('Numero de medicoes')
plt.show()

In [ ]:
# histograma da umidade
plt.figure(figsize=(8, 4))
plt.hist(df['umidade'], bins=15, color='steelblue', edgecolor='black')
plt.title('Distribuicao da Umidade')
plt.xlabel('Umidade (%)')
plt.ylabel('Numero de medicoes')
plt.show()

In [ ]:
# boxplot temperatura por equipamento
plt.figure(figsize=(9, 5))
sns.boxplot(x='equipamento', y='temperatura', data=df, palette='Reds')
plt.title('Temperatura por Equipamento')
plt.xlabel('Equipamento')
plt.ylabel('Temperatura (graus C)')
plt.axhline(35, color='darkred', linestyle='--', label='Limite critico (35 graus C)')
plt.legend()
plt.show()

In [ ]:
# boxplot umidade por equipamento
plt.figure(figsize=(9, 5))
sns.boxplot(x='equipamento', y='umidade', data=df, palette='Blues')
plt.title('Umidade por Equipamento')
plt.xlabel('Equipamento')
plt.ylabel('Umidade (%)')
plt.axhline(70, color='navy', linestyle='--', label='Limite critico (70%)')
plt.legend()
plt.show()

In [ ]:
# dispersao temperatura vs umidade separado por equipamento
plt.figure(figsize=(9, 5))
for equip in df['equipamento'].unique():
    grupo = df[df['equipamento'] == equip]
    plt.scatter(grupo['temperatura'], grupo['umidade'], label=equip, s=60)
plt.title('Temperatura vs Umidade por Equipamento')
plt.xlabel('Temperatura (graus C)')
plt.ylabel('Umidade (%)')
plt.legend()
plt.show()

In [ ]:
# numero de medicoes por Equipamento
plt.figure(figsize=(7, 4))
df['equipamento'].value_counts().plot(kind='bar', color='orange', edgecolor='black')
plt.title('Numero de Medicoes por Equipamento')
plt.xlabel('Equipamento')
plt.ylabel('Quantidade')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# medicoes ao longo dos meses
medicoes_mes = df.groupby('mes').size()
plt.figure(figsize=(7, 4))
medicoes_mes.plot(kind='bar', color='mediumpurple', edgecolor='black')
plt.title('Medicoes por Mes')
plt.xlabel('Mes')
plt.ylabel('Quantidade')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# mapa de correlacao entre variaveis numericas
colunas = ['temperatura', 'umidade', 'equipamento_num', 'sensor_num', 'mes', 'hora']
corr = df[colunas].corr()

plt.figure(figsize=(8, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Mapa de Correlacao entre Variaveis')
plt.show()

## 6. Criando a Variavel de Risco

Para preparar para modelagem, vou criar uma variavel que indica se uma medicao representa situacao de risco.

**Criterio:**
- `risco = 1` se temperatura > 35 graus C **OU** umidade > 70%
- `risco = 0` caso contrario

Esses limites foram definidos com base em boas praticas para operacao de equipamentos eletronicos em ambientes comerciais.

In [ ]:
# criando a variavel risco
df['risco'] = 0
df.loc[(df['temperatura'] > 35) | (df['umidade'] > 70), 'risco'] = 1

print('Distribuicao da variavel risco:')
print(df['risco'].value_counts())
print()
porcentagem = round(df['risco'].mean() * 100, 1)
print(str(porcentagem) + '% das medicoes estao em situacao de risco')

In [ ]:
# percentual de risco por equipamento
risco_equip = df.groupby('equipamento')['risco'].mean() * 100
risco_equip = risco_equip.round(1).sort_values(ascending=False)

cores = ['red' if v >= 60 else 'orange' if v >= 30 else 'green' for v in risco_equip.values]

plt.figure(figsize=(8, 4))
risco_equip.plot(kind='bar', color=cores, edgecolor='black')
plt.title('Percentual de Medicoes em Risco por Equipamento')
plt.xlabel('Equipamento')
plt.ylabel('% em risco')
plt.xticks(rotation=0)
plt.show()

print(risco_equip)

In [ ]:
# medias por equipamento (resumo final)
medias = df.groupby('equipamento')[['temperatura', 'umidade', 'risco']].mean().round(2)
print('Medias por equipamento:')
print(medias)

## 7. Insights Iniciais

Com base na analise exploratoria, os pontos mais importantes sao:

**1. EQ-001 e o equipamento com maior risco termico** — sua temperatura media e a mais alta, frequentemente passando de 35 graus C. E o equipamento que mais precisa de atencao.

**2. EQ-003 tem a maior umidade** — com valores frequentemente acima de 70%, esse equipamento pode ter problemas de condensacao e corrosao nos componentes internos.

**3. EQ-002, EQ-004 e EQ-005** apresentam condicoes mais estaveis, com temperaturas entre 22 e 31 graus C e umidade moderada.

**4. Temperatura e umidade tem baixa correlacao entre si** — sao dois fatores de risco independentes que precisam ser analisados em conjunto.

**5. Ha variacao entre os meses** — o periodo do ano pode influenciar levemente as leituras.

---

### Variaveis que parecem influenciar o risco:

| Variavel | Importancia percebida |
|----------|----------------------|
| temperatura | Alta — EQ-001 muito acima dos demais |
| umidade | Media — EQ-003 em zona critica |
| equipamento | Alta — EQ-001 e EQ-003 sao os mais criticos |
| mes | Baixa — variacao discreta entre meses |

---
*Proxima etapa: aplicar modelo de classificacao (ex: arvore de decisao) para prever risco.*

---

## 8. Preview — Modelagem Preditiva (Entregável 2)

> **Nota:** Esta seção é um **preview antecipado** da fase de Modelagem (CRISP-DM fase 4). O presente entregável (CS02) tem como foco a **Análise Exploratória de Dados (EDA)** e a **Preparação dos Dados**. As células abaixo estão incluídas para demonstrar a aplicabilidade do dataset preparado e dar uma visão do próximo passo.
>
> A análise completa de modelagem — seleção de features, validação cruzada, tuning de hiperparâmetros e avaliação robusta — será desenvolvida no **Entregável 3**.

In [ ]:
# ============================================================
# MODELAGEM — Classificacao de Risco com Arvore de Decisao
# ============================================================
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# preparando features e target
features = ['temperatura', 'umidade', 'equipamento_num', 'sensor_num', 'mes', 'hora']
X = df[features]
y = df['risco']

# divisao treino / teste (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print('Tamanho treino:', X_train.shape)
print('Tamanho teste: ', X_test.shape)
print()
print('Distribuicao de risco no treino:')
print(y_train.value_counts())

# treinando o modelo
clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)

# avaliando no conjunto de teste
y_pred = clf.predict(X_test)

print()
print('=== RESULTADOS DO MODELO ===')
print(f'Acuracia: {accuracy_score(y_test, y_pred) * 100:.1f}%')
print()
print('Relatorio de Classificacao:')
print(classification_report(y_test, y_pred, target_names=['Sem Risco', 'Com Risco']))

# matriz de confusao
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Sem Risco', 'Com Risco'],
            yticklabels=['Sem Risco', 'Com Risco'])
plt.title('Matriz de Confusao - Arvore de Decisao')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

# importancia das variaveis
importancias = pd.Series(clf.feature_importances_, index=features).sort_values(ascending=True)
plt.figure(figsize=(8, 4))
importancias.plot(kind='barh', color='teal', edgecolor='black')
plt.title('Importancia das Variaveis - Arvore de Decisao')
plt.xlabel('Importancia (Gini)')
plt.tight_layout()
plt.show()

print('Ranking de importancia:')
print(importancias.sort_values(ascending=False).to_string())